# 01 - Bronze Ingestion
## Purpose
Connect to the raw layer in ADLS and validate the 911 audio files 
and metadata CSV. No data is modified or saved in this notebook.
Results are used to guide the silver layer processing in notebook 02.

In [0]:
storage_account_name = ""
storage_account_key = ""

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/911-recordings/v1"
print("Connected to ADLS successfully")

Connected to ADLS successfully


In [0]:
metadata_path = f"{base_path}/metadata/911_metadata.csv"

metadata_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(metadata_path)

print("Total metadata rows:", metadata_df.count())
print("Columns:", metadata_df.columns)
display(metadata_df.limit(5))

Total metadata rows: 745
Columns: ['id', 'link', 'title', 'date', 'state', 'civilian_initiated', 'deaths', 'potential_death', 'false_alarm', 'description', 'file_name']


id,link,title,date,state,civilian_initiated,deaths,potential_death,false_alarm,description,file_name
1,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/northhollywood_radio.mp3,North Hollywood bank robbery,2/97,California,0,2,1,0,"– The unforgettable collection of radio logging tapes from the 1997 violent robbery of the Bank of America in Los Angeles. The radio traffic begins routinely, then an officer passing the bank notices the robbers and radios in “shots fired.” Then all breaks loose.",call_1.mp3
2,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/detroit_911_1.mp3,Detroit Child’s 911 Call – audio,2/06,Michigan,1,1,1,0,"– In Feb. 2006 5 year-old Robert Turner called to say his mother was unconscious. However, dispatcher Sharon Nicols believed it was a prank call. Nicols and another dispatcher were later fired, and Nicols was charged with criminal neglect of duty. She was later convicted, but granted probation. Also isten to the second",call_2.mp3
3,https://web.archive.org/web/20150417085342/http://www.msnbc.msn.com/id/8926416/,Sept. 11th Fire Tapes,9/01,null,null,null,null,null,null,null
4,"https://web.archive.org/web/20150417085342/http://www.amny.com/news/local/groundzero/nyc-fdny-mp3,0,6956355.htmlstory",Sept. 11th Fire Tapes #2,9/01,null,null,null,null,null,null,null
5,https://web.archive.org/web/20150417085342/http://www.archive.org/details/911_fdny_dispatches,Sept. 11th Tape Archive,9/01,null,null,null,null,null,null,null


In [0]:
from pyspark.sql.functions import col, when, count as spark_count

print("=== NULL CHECK — METADATA ===")
metadata_df.select([
    spark_count(when(col(c).isNull(), c)).alias(c)
    for c in metadata_df.columns
]).show()

=== NULL CHECK — METADATA ===
+---+----+-----+----+-----+------------------+------+---------------+-----------+-----------+---------+
| id|link|title|date|state|civilian_initiated|deaths|potential_death|false_alarm|description|file_name|
+---+----+-----+----+-----+------------------+------+---------------+-----------+-----------+---------+
|  0|   0|    1|  33|   19|                18|    18|             18|         18|         22|       41|
+---+----+-----+----+-----+------------------+------+---------------+-----------+-----------+---------+



In [0]:
audio_path = f"{base_path}/audio"
audio_files = dbutils.fs.ls(audio_path)
audio_filenames = [f.name for f in audio_files if f.name.endswith(".mp3")]

# Cross check audio vs metadata
metadata_filenames = [
    row["file_name"]
    for row in metadata_df.select("file_name").collect()
    if row["file_name"] is not None
]

in_audio_not_metadata = set(audio_filenames) - set(metadata_filenames)
in_metadata_not_audio = set(metadata_filenames) - set(audio_filenames)
small_files = [f for f in audio_files if f.size < 10000 and f.name.endswith(".mp3")]

print(f"Total audio files         : {len(audio_filenames)}")
print(f"Audio missing metadata    : {len(in_audio_not_metadata)}")
print(f"Metadata missing audio    : {len(in_metadata_not_audio)}")
print(f"Corrupt/small files       : {len(small_files)}")

Total audio files         : 707
Audio missing metadata    : 3
Metadata missing audio    : 0
Corrupt/small files       : 1


In [0]:
# Drop rows with no file_name - cannot be linked to audio
metadata_valid_df = metadata_df.filter(col("file_name").isNotNull())

no_filename = metadata_df.count() - metadata_valid_df.count()
print(f"Rows dropped (no file_name): {no_filename}")
print(f"Valid rows to write        : {metadata_valid_df.count()}")

# Write to processed container
metadata_silver_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/911-recordings/metadata_raw"

metadata_valid_df.write.mode("overwrite").parquet(metadata_silver_path)
print(f"\nSilver layer written: {metadata_silver_path}")

Rows dropped (no file_name): 41
Valid rows to write        : 704

Silver layer written: abfss://processed@azuredatalake60304739.dfs.core.windows.net/911-recordings/metadata_raw


In [0]:
metadata_check = spark.read.parquet(metadata_silver_path)

print("=== SILVER LAYER VERIFICATION ===")
print(f"Rows written : {metadata_check.count()}")
print(f"Columns      : {len(metadata_check.columns)}")
print(f"\nSchema:")
metadata_check.printSchema()
display(metadata_check.limit(5))

=== SILVER LAYER VERIFICATION ===
Rows written : 704
Columns      : 11

Schema:
root
 |-- id: string (nullable = true)
 |-- link: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: string (nullable = true)
 |-- state: string (nullable = true)
 |-- civilian_initiated: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- potential_death: integer (nullable = true)
 |-- false_alarm: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- file_name: string (nullable = true)



id,link,title,date,state,civilian_initiated,deaths,potential_death,false_alarm,description,file_name
1,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/northhollywood_radio.mp3,North Hollywood bank robbery,2/97,California,0,2,1,0,"– The unforgettable collection of radio logging tapes from the 1997 violent robbery of the Bank of America in Los Angeles. The radio traffic begins routinely, then an officer passing the bank notices the robbers and radios in “shots fired.” Then all breaks loose.",call_1.mp3
2,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/detroit_911_1.mp3,Detroit Child’s 911 Call – audio,2/06,Michigan,1,1,1,0,"– In Feb. 2006 5 year-old Robert Turner called to say his mother was unconscious. However, dispatcher Sharon Nicols believed it was a prank call. Nicols and another dispatcher were later fired, and Nicols was charged with criminal neglect of duty. She was later convicted, but granted probation. Also isten to the second",call_2.mp3
8,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/hernlen_choufani_911.mp3,Girl’s Murder 911 Call,3/05,Florida,1,2,1,0,"– the 911 call of a lifetime. Volusia County (Fla.) dispatcher Donna Choufani talks to 5 year-old Tia Hernlen, who calmly reports her two parents shot to death. Also check this",call_8.mp3
9,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/watauga_911.mp3,‘Shoot Her?’ 911 call,4/05,Texas,1,0,0,1,"– caller reports her daughter is creating a disturbance at home, and the dispatcher makes an inappropriate comment",call_9.mp3
10,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/wamsley_to_douglascounty.mp3,Snowstorm 911 Call,1/05,Nebraska,1,2,1,0,– a couple under the influence of drugs dialed 911 after their truck ran into a snowdrift outside Omaha (Neb.) in Jan. 2005. Janelle Hornickel and Michael Wamsley were disoriented and couldn’t give their location. Their bodies were found days later in the snow.,call_10.mp3
